# Module 4: The U-Net Architecture

**Learning Objectives:**
- Understand the U-Net's origin in biomedical segmentation and its adaptation for diffusion models
- Build encoder (downsampling) and decoder (upsampling) paths from scratch
- Implement skip connections and understand why they are essential
- Add sinusoidal timestep conditioning with both additive and scale-shift injection
- Place attention layers at appropriate resolutions
- Assemble a complete, production-quality U-Net for noise prediction

**Estimated time:** 3--4 hours

**Key Papers:**

| Paper | Year | Relevance |
|-------|------|-----------|
| [U-Net -- Ronneberger et al.](https://arxiv.org/abs/1505.04597) | 2015 | Original encoder-decoder + skip connection architecture for segmentation |
| [DDPM -- Ho et al.](https://arxiv.org/abs/2006.11239) | 2020 | Adapted U-Net for diffusion: added timestep conditioning, self-attention, GroupNorm |
| [ADM -- Dhariwal & Nichol](https://arxiv.org/abs/2105.05233) | 2021 | Improved U-Net: Adaptive GroupNorm (AdaGN), attention at multiple resolutions, bigger models |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, List, Tuple

torch.manual_seed(42)

device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

## 4.1 -- U-Net History: From Segmentation to Generation

### The Original U-Net (2015)

The [U-Net](https://arxiv.org/abs/1505.04597) was designed for biomedical image segmentation, where pixel-level accuracy matters. Its core insight: **combine high-level semantic features from deep layers with fine spatial detail from shallow layers** via skip connections.

The architecture has a symmetric encoder-decoder shape (like the letter "U"):

```
Input Image
    |
    v
[Encoder Level 1: 64ch]  ----skip connection---->  [Decoder Level 1: 64ch]
    |  (downsample 2x)                                  ^  (upsample 2x)
    v                                                    |
[Encoder Level 2: 128ch] ----skip connection---->  [Decoder Level 2: 128ch]
    |  (downsample 2x)                                  ^  (upsample 2x)
    v                                                    |
[Encoder Level 3: 256ch] ----skip connection---->  [Decoder Level 3: 256ch]
    |  (downsample 2x)                                  ^  (upsample 2x)
    v                                                    |
              [ Bottleneck: 512ch ]  --------------------+
```

### Why Diffusion Models Adopted the U-Net

Diffusion models need a network that takes a noisy image and predicts the noise (or the clean image). The input and output have **identical spatial dimensions** -- exactly what an encoder-decoder architecture provides naturally.

### Adaptations for Diffusion

| Feature | Original U-Net (Segmentation) | Diffusion U-Net (DDPM/ADM) |
|---------|-------------------------------|---------------------------|
| Normalization | BatchNorm | GroupNorm (works with small batch sizes) |
| Time conditioning | None | Sinusoidal timestep embeddings injected into every ResBlock |
| Attention | None | Self-attention at low-resolution feature maps (16x16, 8x8) |
| Residual connections | No | Yes (within each block) |
| Skip connection type | Crop + Concatenate | Concatenate (same-size feature maps via padding) |
| Block design | 2 plain convolutions | ResBlock with GroupNorm, SiLU activation, optional attention |

## 4.2 -- Encoder Path: Progressive Downsampling

The encoder progressively reduces spatial resolution while increasing channel depth, building a **feature hierarchy**:
- Early layers capture low-level features (edges, textures) at high resolution
- Deeper layers capture high-level features (object parts, structure) at low resolution

Typical channel progression: `64 -> 128 -> 256 -> 512`, with 2x spatial downsampling at each level.

**Downsampling methods:**
- **Stride-2 convolution** (most common in diffusion U-Nets): learns what to discard
- **Average pooling + 1x1 conv**: separates spatial reduction from channel mixing

Each encoder level contains:
1. Two ResBlocks (with time conditioning -- we will add this in Section 4.5)
2. Optional attention (at specific resolutions -- Section 4.6)
3. A downsampling operation

### Worked Example: DownBlock

We start with simplified blocks (no time conditioning or attention yet) to focus on the spatial mechanics.

In [ ]:
class SimpleResBlock(nn.Module):
    """Simplified residual block (no time conditioning yet).
    
    Structure: GroupNorm -> SiLU -> Conv -> GroupNorm -> SiLU -> Conv + residual
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups=8, num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.SiLU()
        
        # 1x1 projection if channel dimensions change
        self.residual_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.residual_proj(x)  # (B, out_channels, H, W)
        x = self.act(self.norm1(x))       # (B, in_channels, H, W)
        x = self.conv1(x)                 # (B, out_channels, H, W)
        x = self.act(self.norm2(x))       # (B, out_channels, H, W)
        x = self.conv2(x)                 # (B, out_channels, H, W)
        return x + residual               # (B, out_channels, H, W)


class SimpleDownBlock(nn.Module):
    """Encoder block: 2 ResBlocks + stride-2 conv for downsampling."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.res1 = SimpleResBlock(in_channels, out_channels)
        self.res2 = SimpleResBlock(out_channels, out_channels)
        self.downsample = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (downsampled_output, skip_connection)."""
        x = self.res1(x)             # (B, out_channels, H, W)
        x = self.res2(x)             # (B, out_channels, H, W)
        skip = x                     # Save for skip connection
        x = self.downsample(x)       # (B, out_channels, H/2, W/2)
        return x, skip


# Test the DownBlock
torch.manual_seed(42)
test_input = torch.randn(2, 64, 32, 32)  # (B, C, H, W)
down_block = SimpleDownBlock(in_channels=64, out_channels=128)
output, skip = down_block(test_input)

print(f"Input shape:  {test_input.shape}")   # (2, 64, 32, 32)
print(f"Skip shape:   {skip.shape}")         # (2, 128, 32, 32) -- same spatial dims, new channels
print(f"Output shape: {output.shape}")       # (2, 128, 16, 16) -- halved spatial dims

### Exercise 4.2: Full Encoder

Chain 4 DownBlocks with increasing channels: `1 -> 64 -> 128 -> 256 -> 512`.
Verify that spatial dimensions halve and channels double at each level. Start from a `(B, 1, 28, 28)` MNIST-sized input (note: 28 is not a power of 2, which introduces rounding -- this is realistic).

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

# Define channel progression
channel_sizes = [64, 128, 256]
initial_conv = nn.Conv2d(1, 64, kernel_size=3, padding=1)  # 1 channel (MNIST) -> 64

# Build encoder
down_blocks = nn.ModuleList()
in_ch = 64
for out_ch in channel_sizes:
    down_blocks.append(SimpleDownBlock(in_ch, out_ch))
    in_ch = out_ch

# Run forward pass and track shapes
x = torch.randn(2, 1, 28, 28)  # (B, 1, 28, 28) MNIST input
print(f"Input:          {x.shape}")

x = initial_conv(x)  # (B, 64, 28, 28)
print(f"After init conv: {x.shape}")

skips = []
for i, block in enumerate(down_blocks):
    x, skip = block(x)
    skips.append(skip)
    print(f"Level {i+1} output: {x.shape}  |  skip: {skip.shape}")

print(f"\nBottleneck: {x.shape}")
print(f"Number of skip connections stored: {len(skips)}")

## 4.3 -- Decoder Path: Upsampling + Skip Connections

The decoder mirrors the encoder, progressively increasing spatial resolution while decreasing channels.

**Upsampling methods:**
- **`F.interpolate(nearest)` + conv** (preferred in modern diffusion models): avoids checkerboard artifacts from transposed convolutions
- **Transposed convolution** (`nn.ConvTranspose2d`): learned upsampling, but prone to checkerboard artifacts

At each decoder level:
1. Upsample the feature map by 2x
2. Concatenate the skip connection from the corresponding encoder level
3. Apply 2 ResBlocks to fuse the information (the first ResBlock handles the doubled channels from concatenation)
4. Optionally apply attention

**Channel math after concatenation:**
If the decoder feature has `D` channels and the skip has `S` channels, concatenation produces `D + S` channels. The first ResBlock in the UpBlock maps `D + S -> out_channels`.

### Worked Example: UpBlock

In [ ]:
class SimpleUpBlock(nn.Module):
    """Decoder block: upsample + concat skip + 2 ResBlocks.
    
    Args:
        in_channels: channels from the previous decoder level (before concat)
        skip_channels: channels from the encoder skip connection
        out_channels: desired output channels
    """
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int):
        super().__init__()
        # Nearest-neighbor upsample + conv (avoids checkerboard artifacts)
        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
        )
        # After concat: in_channels + skip_channels -> out_channels
        self.res1 = SimpleResBlock(in_channels + skip_channels, out_channels)
        self.res2 = SimpleResBlock(out_channels, out_channels)
    
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upsample(x)                        # (B, in_channels, 2H, 2W)
        
        # Handle size mismatch (e.g., 7->14 vs skip at 14, or odd dimensions)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
        
        x = torch.cat([x, skip], dim=1)             # (B, in_channels + skip_channels, 2H, 2W)
        x = self.res1(x)                            # (B, out_channels, 2H, 2W)
        x = self.res2(x)                            # (B, out_channels, 2H, 2W)
        return x


# Test the UpBlock
torch.manual_seed(42)
# Simulate coming from bottleneck: (B, 256, 3, 3) with skip from encoder level 3: (B, 256, 7, 7)
decoder_input = torch.randn(2, 256, 3, 3)   # (B, 256, 3, 3)
skip_connection = torch.randn(2, 256, 7, 7) # (B, 256, 7, 7)

up_block = SimpleUpBlock(in_channels=256, skip_channels=256, out_channels=128)
output = up_block(decoder_input, skip_connection)

print(f"Decoder input:    {decoder_input.shape}")    # (2, 256, 3, 3)
print(f"Skip connection:  {skip_connection.shape}")   # (2, 256, 7, 7)
print(f"After upsample:   (2, 256, 6, 6) -> interpolated to (2, 256, 7, 7)")
print(f"After concat:     (2, 512, 7, 7)  [256 + 256 = 512]")
print(f"Output:           {output.shape}")            # (2, 128, 7, 7)

### Exercise 4.3: Full Decoder

Build the full decoder that mirrors the encoder from Exercise 4.2. Chain 3 UpBlocks with decreasing channels. Use the skip connections stored during encoding. Verify the final output matches the input spatial dimensions `(28, 28)`.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

# Re-run the encoder to get bottleneck + skips
initial_conv_enc = nn.Conv2d(1, 64, kernel_size=3, padding=1)
encoder_channels = [64, 128, 256]
encoder_blocks = nn.ModuleList()
in_ch = 64
for out_ch in encoder_channels:
    encoder_blocks.append(SimpleDownBlock(in_ch, out_ch))
    in_ch = out_ch

x = torch.randn(2, 1, 28, 28)  # (B, 1, 28, 28)
x = initial_conv_enc(x)         # (B, 64, 28, 28)

skips = []
for block in encoder_blocks:
    x, skip = block(x)
    skips.append(skip)

print("=== Encoder outputs ===")
for i, s in enumerate(skips):
    print(f"  Skip {i}: {s.shape}")
print(f"  Bottleneck: {x.shape}")

# Build decoder (mirror of encoder)
# Decoder goes: 256->128->64, consuming skips in reverse order
decoder_configs = [
    (256, 256, 128),  # (in_channels, skip_channels, out_channels) -- level 3 skip is 256ch
    (128, 128, 64),   # level 2 skip is 128ch
    (64, 64, 64),     # level 1 skip is 64ch
]
decoder_blocks = nn.ModuleList()
for in_ch, skip_ch, out_ch in decoder_configs:
    decoder_blocks.append(SimpleUpBlock(in_ch, skip_ch, out_ch))

# Final projection back to image channels
final_conv = nn.Sequential(
    nn.GroupNorm(8, 64),
    nn.SiLU(),
    nn.Conv2d(64, 1, kernel_size=3, padding=1),
)

# Run decoder
print("\n=== Decoder ===")
for i, block in enumerate(decoder_blocks):
    skip = skips.pop()  # Pop from the end (last encoder skip = first decoder input)
    x = block(x, skip)
    print(f"  Level {i+1} output: {x.shape}")

x = final_conv(x)  # (B, 1, 28, 28)
print(f"\nFinal output: {x.shape}")
print(f"Matches input spatial dims (28, 28): {x.shape[-2:] == (28, 28)}")

## 4.4 -- Skip Connections: Why They Matter

Skip connections are the defining feature that makes a U-Net a U-Net rather than a plain autoencoder.

**Without skip connections:** The decoder must reconstruct all spatial detail from the compressed bottleneck representation alone. Fine details (edges, textures, precise positions) are lost during encoding and cannot be recovered.

**With skip connections:** Fine-grained spatial information flows directly from encoder to decoder at each resolution level. The decoder only needs to learn *what to change*, not reconstruct everything from scratch.

**Why concatenation over addition:**
- **Concatenation** (used in DDPM): the decoder sees both its own features AND the encoder features separately, then learns to combine them. More expressive -- the network can learn to weight them differently.
- **Addition**: forces the encoder and decoder features into the same representational space. Simpler but less flexible.

**Two additional benefits:**
1. **Gradient highway**: gradients flow directly from decoder loss back to encoder layers, avoiding vanishing gradient problems (same principle as ResNet skip connections)
2. **Memory cost**: encoder feature maps must be stored during the forward pass until the corresponding decoder level consumes them. For large images, this dominates GPU memory usage.

### Worked Example: Denoising With vs. Without Skip Connections

We train two tiny U-Nets on a simple denoising task to visualize the difference.

In [ ]:
import matplotlib.pyplot as plt

class TinyUNet(nn.Module):
    """Minimal U-Net for demonstrating skip connection importance."""
    def __init__(self, use_skip: bool = True):
        super().__init__()
        self.use_skip = use_skip
        
        # Encoder
        self.enc1 = SimpleResBlock(1, 32)
        self.down1 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.enc2 = SimpleResBlock(32, 64)
        self.down2 = nn.Conv2d(64, 64, 3, stride=2, padding=1)
        
        # Bottleneck
        self.bottleneck = SimpleResBlock(64, 64)
        
        # Decoder
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                  nn.Conv2d(64, 64, 3, padding=1))
        # If skip: concat adds 64 channels from encoder. If no skip: just 64 channels.
        self.dec2 = SimpleResBlock(64 + (64 if use_skip else 0), 32)
        
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                  nn.Conv2d(32, 32, 3, padding=1))
        self.dec1 = SimpleResBlock(32 + (32 if use_skip else 0), 32)
        
        self.final = nn.Conv2d(32, 1, 1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        e1 = self.enc1(x)                  # (B, 32, H, W)
        e2 = self.enc2(self.down1(e1))     # (B, 64, H/2, W/2)
        
        # Bottleneck
        b = self.bottleneck(self.down2(e2))  # (B, 64, H/4, W/4)
        
        # Decoder
        d2 = self.up2(b)                   # (B, 64, H/2, W/2)
        if d2.shape[-2:] != e2.shape[-2:]:
            d2 = F.interpolate(d2, size=e2.shape[-2:], mode='nearest')
        if self.use_skip:
            d2 = torch.cat([d2, e2], dim=1)  # (B, 128, H/2, W/2)
        d2 = self.dec2(d2)                 # (B, 32, H/2, W/2)
        
        d1 = self.up1(d2)                  # (B, 32, H, W)
        if d1.shape[-2:] != e1.shape[-2:]:
            d1 = F.interpolate(d1, size=e1.shape[-2:], mode='nearest')
        if self.use_skip:
            d1 = torch.cat([d1, e1], dim=1)  # (B, 64, H, W)
        d1 = self.dec1(d1)                 # (B, 32, H, W)
        
        return self.final(d1)              # (B, 1, H, W)


def train_denoiser(model: nn.Module, num_steps: int = 500) -> List[float]:
    """Train on synthetic denoising: predict clean from noisy."""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    
    for step in range(num_steps):
        # Create simple patterns (horizontal + vertical bars)
        clean = torch.zeros(8, 1, 28, 28)
        for i in range(8):
            # Random bars
            h_pos = torch.randint(0, 28, (3,))
            v_pos = torch.randint(0, 28, (3,))
            clean[i, 0, h_pos, :] = 1.0
            clean[i, 0, :, v_pos] = 1.0
        
        noise = torch.randn_like(clean) * 0.5
        noisy = clean + noise
        
        pred = model(noisy)
        loss = F.mse_loss(pred, clean)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses


# Train both models
torch.manual_seed(42)
model_with_skip = TinyUNet(use_skip=True)
losses_with = train_denoiser(model_with_skip, num_steps=300)

torch.manual_seed(42)
model_no_skip = TinyUNet(use_skip=False)
losses_no = train_denoiser(model_no_skip, num_steps=300)

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(losses_with, label='With skips', alpha=0.7)
axes[0].plot(losses_no, label='Without skips', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

# Visualize denoising quality
torch.manual_seed(99)
clean_test = torch.zeros(1, 1, 28, 28)
clean_test[0, 0, [5, 12, 20], :] = 1.0
clean_test[0, 0, :, [8, 15, 22]] = 1.0
noisy_test = clean_test + torch.randn_like(clean_test) * 0.5

with torch.no_grad():
    pred_with = model_with_skip(noisy_test)
    pred_no = model_no_skip(noisy_test)

axes[1].imshow(pred_with[0, 0].clamp(0, 1), cmap='gray')
axes[1].set_title('With Skip Connections')
axes[1].axis('off')

axes[2].imshow(pred_no[0, 0].clamp(0, 1), cmap='gray')
axes[2].set_title('Without Skip Connections')
axes[2].axis('off')

plt.suptitle('Skip connections preserve spatial detail during denoising', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Final loss WITH skips:    {losses_with[-1]:.4f}")
print(f"Final loss WITHOUT skips: {losses_no[-1]:.4f}")

## 4.5 -- Time Conditioning: Sinusoidal Timestep Embeddings

The diffusion U-Net must behave differently depending on the noise level. At `t=1` (nearly clean), it should make tiny corrections. At `t=999` (nearly pure noise), it must hallucinate large-scale structure. The model receives the timestep `t` as input and conditions its behavior on it.

### Step 1: Sinusoidal Embedding

The same formula used for positional encoding in Transformers ([Vaswani et al. 2017](https://arxiv.org/abs/1706.03762)):

$$\text{emb}[2i] = \sin\left(\frac{t}{10000^{2i/d}}\right), \quad \text{emb}[2i+1] = \cos\left(\frac{t}{10000^{2i/d}}\right)$$

This maps each scalar timestep to a high-dimensional vector where nearby timesteps have similar embeddings and distant timesteps are easily distinguishable.

### Step 2: MLP Projection

The sinusoidal embedding is projected through a small MLP: `Linear -> SiLU -> Linear`. This allows the network to learn a richer timestep representation.

### Step 3: Injection into ResBlocks

Two methods:
- **Additive injection**: project `time_emb` to match the channel dimension, reshape to `(B, C, 1, 1)`, and add to the feature map (broadcasting over spatial dims). Simple and effective -- used in DDPM.
- **Scale-shift (AdaGN)**: project `time_emb` to predict both scale and shift parameters: `gamma, beta = chunk(Linear(time_emb), 2)`. Then apply: `y = gamma * GroupNorm(x) + beta`. More expressive -- used in [ADM (Dhariwal & Nichol 2021)](https://arxiv.org/abs/2105.05233).

### Worked Example

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    """Maps integer timesteps to sinusoidal embeddings.
    
    Args:
        embedding_dim: dimension of the output embedding vector
    """
    def __init__(self, embedding_dim: int):
        super().__init__()
        self.embedding_dim = embedding_dim
    
    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """
        Args:
            timesteps: (B,) integer tensor of timesteps
        Returns:
            embeddings: (B, embedding_dim) float tensor
        """
        half_dim = self.embedding_dim // 2
        # Compute frequencies: 1 / 10000^(2i/d)
        frequencies = torch.exp(
            -math.log(10000.0) * torch.arange(half_dim, device=timesteps.device) / half_dim
        )  # (half_dim,)
        
        # Outer product: each timestep multiplied by each frequency
        angles = timesteps[:, None].float() * frequencies[None, :]  # (B, half_dim)
        
        # Interleave sin and cos
        embeddings = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B, embedding_dim)
        return embeddings


# Test sinusoidal embeddings
sin_emb = SinusoidalTimestepEmbedding(embedding_dim=128)
test_timesteps = torch.tensor([0, 1, 50, 500, 999])  # (5,)
embeddings = sin_emb(test_timesteps)
print(f"Input timesteps: {test_timesteps.shape}")    # (5,)
print(f"Output embeddings: {embeddings.shape}")      # (5, 128)

# Visualize: nearby timesteps should have similar embeddings
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Show embedding vectors as heatmap
axes[0].imshow(embeddings.detach().numpy(), aspect='auto', cmap='RdBu')
axes[0].set_yticks(range(5))
axes[0].set_yticklabels([f't={t.item()}' for t in test_timesteps])
axes[0].set_xlabel('Embedding dimension')
axes[0].set_title('Sinusoidal Timestep Embeddings')
axes[0].set_ylabel('Timestep')

# Show pairwise cosine similarity
all_t = torch.arange(0, 1000, 10)
all_emb = sin_emb(all_t)
cos_sim = F.cosine_similarity(all_emb[:, None, :], all_emb[None, :, :], dim=-1)
axes[1].imshow(cos_sim.detach().numpy(), cmap='viridis', aspect='auto')
axes[1].set_xlabel('Timestep (x10)')
axes[1].set_ylabel('Timestep (x10)')
axes[1].set_title('Cosine Similarity Between Timestep Embeddings')

plt.tight_layout()
plt.show()
print("Nearby timesteps have high similarity; distant timesteps are distinguishable.")

In [ ]:
class TimestepMLPEmbedding(nn.Module):
    """Full timestep embedding pipeline: sinusoidal -> MLP projection.
    
    Args:
        time_embed_dim: dimension of the sinusoidal embedding AND the output
    """
    def __init__(self, time_embed_dim: int):
        super().__init__()
        self.sinusoidal = SinusoidalTimestepEmbedding(time_embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(time_embed_dim, time_embed_dim * 4),
            nn.SiLU(),
            nn.Linear(time_embed_dim * 4, time_embed_dim),
        )
    
    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """
        Args:
            timesteps: (B,) integer tensor
        Returns:
            time_emb: (B, time_embed_dim) projected embedding
        """
        emb = self.sinusoidal(timesteps)   # (B, time_embed_dim)
        return self.mlp(emb)               # (B, time_embed_dim)


# Demonstrate both injection methods
torch.manual_seed(42)
time_embed_dim = 128
time_mlp = TimestepMLPEmbedding(time_embed_dim)
t = torch.tensor([10, 500])  # (2,) batch of 2 timesteps
time_emb = time_mlp(t)       # (2, 128)
print(f"Time embedding shape: {time_emb.shape}")

# --- Method 1: Additive injection ---
feature_channels = 64
time_proj_add = nn.Linear(time_embed_dim, feature_channels)
feature_map = torch.randn(2, feature_channels, 16, 16)  # (B, C, H, W)

projected = time_proj_add(time_emb)          # (B, 64)
projected = projected[:, :, None, None]       # (B, 64, 1, 1) -- broadcast-ready
result_add = feature_map + projected          # (B, 64, 16, 16) -- broadcast over H, W

print(f"\nAdditive injection:")
print(f"  Feature map:    {feature_map.shape}")
print(f"  Time projected: {projected.shape}  (broadcast over spatial dims)")
print(f"  Result:         {result_add.shape}")

# --- Method 2: Scale-shift (AdaGN) injection ---
time_proj_adagn = nn.Linear(time_embed_dim, feature_channels * 2)  # predict gamma AND beta
group_norm = nn.GroupNorm(8, feature_channels)

scale_shift = time_proj_adagn(time_emb)                  # (B, 128)
scale, shift = scale_shift.chunk(2, dim=-1)               # (B, 64), (B, 64)
scale = scale[:, :, None, None]                           # (B, 64, 1, 1)
shift = shift[:, :, None, None]                           # (B, 64, 1, 1)

normalized = group_norm(feature_map)                      # (B, 64, 16, 16)
result_adagn = (1 + scale) * normalized + shift           # (B, 64, 16, 16)
# Note: (1 + scale) so that initialization near zero gives identity-like behavior

print(f"\nScale-shift (AdaGN) injection:")
print(f"  Scale:  {scale.shape}  (per-channel, broadcast spatial)")
print(f"  Shift:  {shift.shape}")
print(f"  Result: {result_adagn.shape}")

### Exercise 4.5: Full Timestep Embedding Pipeline

Implement a `ResBlockWithTime` that takes an integer timestep through the full pipeline: `int t -> sinusoidal -> MLP -> inject into ResBlock`. Use additive injection. Verify that the same spatial input produces different outputs for different timesteps.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
class ResBlockWithTime(nn.Module):
    """ResBlock with additive time conditioning.
    
    Structure: GroupNorm -> SiLU -> Conv -> +time -> GroupNorm -> SiLU -> Conv + residual
    Time is injected after the first conv+norm+act, before the second.
    """
    def __init__(self, in_channels: int, out_channels: int, time_embed_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups=min(8, in_channels), num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=min(8, out_channels), num_channels=out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.SiLU()
        
        # Time projection: time_embed_dim -> out_channels
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_embed_dim, out_channels),
        )
        
        # Residual projection if channels change
        self.residual_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )
    
    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, in_channels, H, W)
            time_emb: (B, time_embed_dim) -- already projected through MLP
        Returns:
            (B, out_channels, H, W)
        """
        residual = self.residual_proj(x)        # (B, out_channels, H, W)
        
        x = self.act(self.norm1(x))             # (B, in_channels, H, W)
        x = self.conv1(x)                       # (B, out_channels, H, W)
        
        # Inject time: project and add (broadcast over spatial dims)
        t = self.time_proj(time_emb)            # (B, out_channels)
        x = x + t[:, :, None, None]             # (B, out_channels, H, W)
        
        x = self.act(self.norm2(x))             # (B, out_channels, H, W)
        x = self.conv2(x)                       # (B, out_channels, H, W)
        
        return x + residual                     # (B, out_channels, H, W)


# Test: same input, different timesteps -> different outputs
torch.manual_seed(42)
time_embed_dim = 128
time_mlp = TimestepMLPEmbedding(time_embed_dim)
res_block = ResBlockWithTime(64, 128, time_embed_dim)

x = torch.randn(1, 64, 16, 16)  # (1, 64, 16, 16)

t_early = torch.tensor([1])
t_late = torch.tensor([999])

time_emb_early = time_mlp(t_early)  # (1, 128)
time_emb_late = time_mlp(t_late)    # (1, 128)

with torch.no_grad():
    out_early = res_block(x, time_emb_early)  # (1, 128, 16, 16)
    out_late = res_block(x, time_emb_late)    # (1, 128, 16, 16)

print(f"Input shape:          {x.shape}")
print(f"Output shape (t=1):   {out_early.shape}")
print(f"Output shape (t=999): {out_late.shape}")
print(f"Outputs differ:       {not torch.allclose(out_early, out_late)}")
print(f"Mean abs difference:  {(out_early - out_late).abs().mean():.4f}")

## 4.6 -- Where Attention Goes in the U-Net

Self-attention lets every spatial location attend to every other spatial location, capturing long-range dependencies that convolutions (with their limited receptive fields) cannot. However, attention has **O(n^2)** cost in the number of spatial tokens, making resolution placement critical.

### Compute Cost at Different Resolutions

For a feature map of size `(H, W)`, the number of tokens is `n = H * W`. Self-attention computes an `n x n` attention matrix, so:

| Resolution | Tokens (n = H*W) | Attention cost (n^2) | Relative cost |
|------------|-------------------|----------------------|---------------|
| 64 x 64   | 4,096             | 16,777,216           | 1024x         |
| 32 x 32   | 1,024             | 1,048,576            | 64x           |
| 16 x 16   | 256               | 65,536               | 4x            |
| 8 x 8     | 64                | 4,096                | 0.25x         |
| 4 x 4     | 16                | 256                  | 0.016x        |

Placing attention at 64x64 is **1024x more expensive** than at 8x8. This is why diffusion models restrict attention to low-resolution feature maps:

- **DDPM** (Ho et al. 2020): attention at **16x16 only**
- **ADM** (Dhariwal & Nichol 2021): attention at **32x32, 16x16, and 8x8**

At low resolutions, each spatial position already encodes a large receptive field from the preceding convolutions, so attention efficiently combines high-level semantic features across the image.

### Implementation: AttentionBlock

The standard spatial self-attention block for U-Nets:
1. GroupNorm the input
2. Project to Q, K, V with 1x1 convolutions
3. Reshape `(B, C, H, W)` to `(B, H*W, C)` for attention computation
4. Scaled dot-product attention
5. Project output back and add residual connection

In [ ]:
class AttentionBlock(nn.Module):
    """Spatial self-attention block for U-Net feature maps.
    
    Applies multi-head self-attention over spatial positions with a residual
    connection. Designed to be placed after ResBlocks at low-resolution levels.
    
    Args:
        channels: number of input (and output) channels
        num_heads: number of attention heads (channels must be divisible by num_heads)
    """
    def __init__(self, channels: int, num_heads: int = 1):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        assert channels % num_heads == 0, f"channels ({channels}) must be divisible by num_heads ({num_heads})"
        self.head_dim = channels // num_heads
        
        self.norm = nn.GroupNorm(num_groups=min(8, channels), num_channels=channels)
        
        # QKV projections using 1x1 convolutions
        self.to_qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        
        # Output projection
        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)
        
        # Initialize output projection to zero for stable training
        nn.init.zeros_(self.proj_out.weight)
        nn.init.zeros_(self.proj_out.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, C, H, W) feature map
        Returns:
            (B, C, H, W) feature map with self-attention applied + residual
        """
        B, C, H, W = x.shape
        residual = x                                          # (B, C, H, W)
        
        x = self.norm(x)                                      # (B, C, H, W)
        qkv = self.to_qkv(x)                                  # (B, 3*C, H, W)
        qkv = qkv.reshape(B, 3, self.num_heads, self.head_dim, H * W)  # (B, 3, heads, head_dim, n)
        qkv = qkv.permute(1, 0, 2, 4, 3)                     # (3, B, heads, n, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]                     # each: (B, heads, n, head_dim)
        
        # Scaled dot-product attention
        scale = self.head_dim ** -0.5
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, heads, n, n)
        attn_weights = F.softmax(attn_weights, dim=-1)                # (B, heads, n, n)
        
        out = torch.matmul(attn_weights, v)                   # (B, heads, n, head_dim)
        out = out.permute(0, 1, 3, 2)                         # (B, heads, head_dim, n)
        out = out.reshape(B, C, H, W)                         # (B, C, H, W)
        
        out = self.proj_out(out)                               # (B, C, H, W)
        return out + residual                                  # (B, C, H, W)


# Test AttentionBlock
torch.manual_seed(42)
attn_block = AttentionBlock(channels=128, num_heads=4)

# Test at different resolutions and measure compute time
import time

for res in [4, 7, 14, 28]:
    test_x = torch.randn(2, 128, res, res)  # (2, 128, res, res)
    
    start = time.time()
    with torch.no_grad():
        out = attn_block(test_x)
    elapsed = (time.time() - start) * 1000
    
    n_tokens = res * res
    print(f"Resolution {res:2d}x{res:2d} | tokens: {n_tokens:5d} | "
          f"attn matrix: {n_tokens:5d}x{n_tokens:<5d} | "
          f"output: {out.shape} | time: {elapsed:.1f}ms")

print(f"\nAttentionBlock parameters: {sum(p.numel() for p in attn_block.parameters()):,}")

## 4.7 -- ResBlock with Time Embedding: Full Implementation

In Section 4.5 we built a basic `ResBlockWithTime`. Now we build the **production version** that matches DDPM/ADM implementations more closely. The key refinement is the ordering: we apply GroupNorm and activation *before* each convolution (pre-activation style), with time injection between the two conv layers.

**Full structure:**
```
x -> GroupNorm -> SiLU -> Conv3x3 -> (+time_proj) -> GroupNorm -> SiLU -> Dropout -> Conv3x3 -> (+residual)
```

The `ResBlock` below supports additive time injection and optional dropout for regularization. It also cleanly handles channel mismatches via a 1x1 residual projection.

In [ ]:
class ResBlock(nn.Module):
    """Production residual block with time conditioning and dropout.
    
    Structure: GroupNorm -> SiLU -> Conv -> +time -> GroupNorm -> SiLU -> Dropout -> Conv + residual
    
    Args:
        in_channels: input feature channels
        out_channels: output feature channels
        time_embed_dim: dimension of the time embedding vector
        dropout: dropout rate applied before the second convolution
    """
    def __init__(self, in_channels: int, out_channels: int, time_embed_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups=min(32, in_channels), num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=min(32, out_channels), num_channels=out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        
        # Time projection: time_embed_dim -> out_channels (additive injection)
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_embed_dim, out_channels),
        )
        
        # 1x1 residual projection if channel dimensions change
        self.residual_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )
        
        # Initialize second conv near zero for stable training
        nn.init.zeros_(self.conv2.weight)
        nn.init.zeros_(self.conv2.bias)
    
    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, in_channels, H, W)
            time_emb: (B, time_embed_dim) -- already projected through timestep MLP
        Returns:
            (B, out_channels, H, W)
        """
        residual = self.residual_proj(x)        # (B, out_channels, H, W)
        
        x = self.act(self.norm1(x))             # (B, in_channels, H, W)
        x = self.conv1(x)                       # (B, out_channels, H, W)
        
        # Additive time injection (broadcast over spatial dims)
        t = self.time_proj(time_emb)            # (B, out_channels)
        x = x + t[:, :, None, None]             # (B, out_channels, H, W)
        
        x = self.act(self.norm2(x))             # (B, out_channels, H, W)
        x = self.dropout(x)                     # (B, out_channels, H, W)
        x = self.conv2(x)                       # (B, out_channels, H, W)
        
        return x + residual                     # (B, out_channels, H, W)


# Test with matching channels
torch.manual_seed(42)
time_dim = 128
time_mlp_test = TimestepMLPEmbedding(time_dim)

res_same = ResBlock(in_channels=64, out_channels=64, time_embed_dim=time_dim)
x_test = torch.randn(2, 64, 14, 14)         # (2, 64, 14, 14)
t_test = torch.tensor([100, 500])            # (2,)
t_emb_test = time_mlp_test(t_test)           # (2, 128)

with torch.no_grad():
    out_same = res_same(x_test, t_emb_test)  # (2, 64, 14, 14)
print(f"Matching channels:   {x_test.shape} -> {out_same.shape}")

# Test with mismatched channels
res_diff = ResBlock(in_channels=64, out_channels=128, time_embed_dim=time_dim, dropout=0.1)
with torch.no_grad():
    out_diff = res_diff(x_test, t_emb_test)  # (2, 128, 14, 14)
print(f"Mismatched channels: {x_test.shape} -> {out_diff.shape}")

# Verify time conditioning works
t_emb_a = time_mlp_test(torch.tensor([1]))    # (1, 128)
t_emb_b = time_mlp_test(torch.tensor([999]))  # (1, 128)
x_single = torch.randn(1, 64, 14, 14)         # (1, 64, 14, 14)

res_same.eval()
with torch.no_grad():
    out_a = res_same(x_single, t_emb_a)       # (1, 64, 14, 14)
    out_b = res_same(x_single, t_emb_b)       # (1, 64, 14, 14)

print(f"\nSame input, t=1 vs t=999 differ: {not torch.allclose(out_a, out_b)}")
print(f"Mean abs difference: {(out_a - out_b).abs().mean():.4f}")
print(f"\nResBlock params (64->64):  {sum(p.numel() for p in res_same.parameters()):,}")
print(f"ResBlock params (64->128): {sum(p.numel() for p in res_diff.parameters()):,}")

## 4.8 -- Assembling the Full U-Net

Now we bring together every component built so far -- `ResBlock`, `AttentionBlock`, `SinusoidalTimestepEmbedding` -- into a complete, production-quality U-Net for diffusion models.

### Architecture Overview

```
Input (B, 1, 28, 28)
    |
    v
[Initial Conv: 1 -> 64]                           (B, 64, 28, 28)
    |
    v
[DownBlock 1: 64ch, 2 ResBlocks]  --skip x2-->    (B, 64, 14, 14)
    |
    v
[DownBlock 2: 128ch, 2 ResBlocks] --skip x2-->    (B, 128, 7, 7)  <-- attention here
    |
    v
[DownBlock 3: 256ch, 2 ResBlocks] --skip x2-->    (B, 256, 3, 3)
    |
    v
[Middle: ResBlock + Attention + ResBlock]          (B, 256, 3, 3)
    |
    v
[UpBlock 3: concat skip, 256->256]                (B, 256, 7, 7)
    |
    v
[UpBlock 2: concat skip, 128->128] <-- attn       (B, 128, 14, 14)
    |
    v
[UpBlock 1: concat skip, 64->64]                  (B, 64, 28, 28)
    |
    v
[Final: GroupNorm -> SiLU -> Conv 64->1]           (B, 1, 28, 28)
```

### Key Design Decisions

- **`attention_resolutions`**: tuple of spatial resolutions where attention is applied (e.g., `(7,)` for 7x7 feature maps in our MNIST setup)
- **`num_res_blocks`**: number of ResBlocks per encoder/decoder level (DDPM uses 2)
- **Skip connections**: each ResBlock output in the encoder is stored as a skip; the decoder concatenates and processes them
- **Class conditioning (optional)**: if `num_classes` is provided, a class embedding is added to the time embedding for class-conditional generation

### DownBlock and UpBlock with Time + Attention

We need updated DownBlock/UpBlock that accept time embeddings and optionally include attention.

In [ ]:
class DownBlock(nn.Module):
    """Encoder block: N ResBlocks (with time) + optional attention + downsample.
    
    Each ResBlock output is stored as a skip connection for the decoder.
    The downsampled output is also stored as a skip (consumed first by the decoder).
    
    Args:
        in_channels: input channels
        out_channels: output channels
        time_embed_dim: dimension of time embedding
        num_res_blocks: number of ResBlocks in this level
        use_attention: whether to apply attention after each ResBlock
        dropout: dropout rate
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        time_embed_dim: int,
        num_res_blocks: int = 2,
        use_attention: bool = False,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.res_blocks = nn.ModuleList()
        self.attn_blocks = nn.ModuleList()
        
        for i in range(num_res_blocks):
            ch_in = in_channels if i == 0 else out_channels
            self.res_blocks.append(ResBlock(ch_in, out_channels, time_embed_dim, dropout))
            self.attn_blocks.append(
                AttentionBlock(out_channels) if use_attention else nn.Identity()
            )
        
        self.downsample = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1)
    
    def forward(
        self, x: torch.Tensor, time_emb: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Args:
            x: (B, in_channels, H, W)
            time_emb: (B, time_embed_dim)
        Returns:
            (downsampled_output, list_of_skip_connections)
        """
        skips = []
        for res_block, attn_block in zip(self.res_blocks, self.attn_blocks):
            x = res_block(x, time_emb)      # (B, out_channels, H, W)
            x = attn_block(x)               # (B, out_channels, H, W)
            skips.append(x)
        
        x = self.downsample(x)              # (B, out_channels, H//2, W//2)
        skips.append(x)
        return x, skips


class UpBlock(nn.Module):
    """Decoder block: upsample + N ResBlocks (with time + skip concat) + optional attention.
    
    Consumes skip connections from the corresponding DownBlock in reverse order.
    
    Args:
        in_channels: channels coming from the previous (deeper) decoder level
        out_channels: desired output channels
        skip_channels: channels from the encoder skip connection
        time_embed_dim: dimension of time embedding
        num_res_blocks: number of ResBlocks in this level
        use_attention: whether to apply attention after each ResBlock
        dropout: dropout rate
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        skip_channels: int,
        time_embed_dim: int,
        num_res_blocks: int = 2,
        use_attention: bool = False,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        )
        
        self.res_blocks = nn.ModuleList()
        self.attn_blocks = nn.ModuleList()
        
        for i in range(num_res_blocks):
            # First ResBlock takes concat of upsampled + skip
            ch_in = (out_channels + skip_channels) if i == 0 else out_channels
            self.res_blocks.append(ResBlock(ch_in, out_channels, time_embed_dim, dropout))
            self.attn_blocks.append(
                AttentionBlock(out_channels) if use_attention else nn.Identity()
            )
    
    def forward(
        self, x: torch.Tensor, skips: List[torch.Tensor], time_emb: torch.Tensor
    ) -> torch.Tensor:
        """
        Args:
            x: (B, in_channels, H, W) from deeper decoder level
            skips: list of skip tensors from the encoder (consumed in reverse)
            time_emb: (B, time_embed_dim)
        Returns:
            (B, out_channels, 2H, 2W)
        """
        x = self.upsample(x)                    # (B, out_channels, 2H, 2W)
        
        for i, (res_block, attn_block) in enumerate(zip(self.res_blocks, self.attn_blocks)):
            skip = skips.pop()                   # Pop the most recent skip
            # Handle spatial size mismatch
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
            if i == 0:
                x = torch.cat([x, skip], dim=1) # (B, out_channels + skip_channels, H, W)
            else:
                x = x + skip                    # Additive merge for subsequent blocks
            x = res_block(x, time_emb)          # (B, out_channels, H, W)
            x = attn_block(x)                   # (B, out_channels, H, W)
        
        return x


# Quick test of DownBlock and UpBlock
torch.manual_seed(42)
time_dim = 128
temb = torch.randn(2, time_dim)  # (2, 128) dummy time embedding

down = DownBlock(64, 128, time_dim, num_res_blocks=2, use_attention=True)
x_in = torch.randn(2, 64, 14, 14)  # (2, 64, 14, 14)
x_out, skip_list = down(x_in, temb)

print(f"DownBlock: {x_in.shape} -> {x_out.shape}")
print(f"  Skips: {[s.shape for s in skip_list]}")

up = UpBlock(128, 64, 128, time_dim, num_res_blocks=2, use_attention=True)
x_up = up(x_out, skip_list, temb)
print(f"UpBlock:   {x_out.shape} -> {x_up.shape}")
print(f"  Remaining skips: {len(skip_list)}")

In [ ]:
class UNet(nn.Module):
    """Complete U-Net for diffusion models with time and optional class conditioning.
    
    Architecture: initial_conv -> [DownBlocks] -> middle -> [UpBlocks] -> final_conv
    
    Args:
        image_channels: number of input/output image channels (1 for MNIST, 3 for RGB)
        base_channels: base channel count (doubled at each level via channel_mults)
        channel_mults: multipliers for channel count at each encoder/decoder level
        num_res_blocks: number of ResBlocks per encoder/decoder level
        attention_resolutions: tuple of spatial resolutions where attention is applied
        dropout: dropout rate in ResBlocks
        num_classes: if provided, enables class-conditional generation with this many classes
    """
    def __init__(
        self,
        image_channels: int = 1,
        base_channels: int = 64,
        channel_mults: Tuple[int, ...] = (1, 2, 4),
        num_res_blocks: int = 2,
        attention_resolutions: Tuple[int, ...] = (7,),
        dropout: float = 0.0,
        num_classes: Optional[int] = None,
    ):
        super().__init__()
        self.image_channels = image_channels
        self.num_classes = num_classes
        time_embed_dim = base_channels * 4
        
        # --- Time embedding ---
        self.time_embedding = nn.Sequential(
            SinusoidalTimestepEmbedding(base_channels),
            nn.Linear(base_channels, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim),
        )
        
        # --- Optional class embedding ---
        if num_classes is not None:
            # +1 for unconditional class (used in classifier-free guidance)
            self.class_embedding = nn.Embedding(num_classes + 1, time_embed_dim)
        else:
            self.class_embedding = None
        
        # --- Initial convolution ---
        self.initial_conv = nn.Conv2d(image_channels, base_channels, kernel_size=3, padding=1)
        
        # --- Encoder (down) path ---
        self.down_blocks = nn.ModuleList()
        channels = [base_channels]  # Track channels at each level for skip connections
        ch_in = base_channels
        current_res = 28  # Starting resolution (will be updated per level)
        
        for level, mult in enumerate(channel_mults):
            ch_out = base_channels * mult
            use_attn = (current_res // 2) in attention_resolutions  # Attention at output res
            self.down_blocks.append(
                DownBlock(
                    in_channels=ch_in,
                    out_channels=ch_out,
                    time_embed_dim=time_embed_dim,
                    num_res_blocks=num_res_blocks,
                    use_attention=use_attn,
                    dropout=dropout,
                )
            )
            ch_in = ch_out
            current_res = current_res // 2
            channels.extend([ch_out] * num_res_blocks + [ch_out])  # ResBlock outputs + downsample
        
        # --- Middle block ---
        self.middle_res1 = ResBlock(ch_in, ch_in, time_embed_dim, dropout)
        self.middle_attn = AttentionBlock(ch_in)
        self.middle_res2 = ResBlock(ch_in, ch_in, time_embed_dim, dropout)
        
        # --- Decoder (up) path ---
        self.up_blocks = nn.ModuleList()
        
        for level in reversed(range(len(channel_mults))):
            mult = channel_mults[level]
            ch_out = base_channels * mult
            # Determine skip channels: pop from the channels list
            skip_ch = channels.pop()  # The downsample skip
            
            # Determine attention: mirror encoder
            prev_mult = channel_mults[level - 1] if level > 0 else 1
            ch_from_deeper = base_channels * channel_mults[min(level + 1, len(channel_mults) - 1)] if level < len(channel_mults) - 1 else ch_in
            
            # Resolution at this decoder level output
            decoder_res = 28 // (2 ** level)
            use_attn = decoder_res in attention_resolutions
            
            self.up_blocks.append(
                UpBlock(
                    in_channels=ch_in if level == len(channel_mults) - 1 else ch_in,
                    out_channels=ch_out,
                    skip_channels=skip_ch,
                    time_embed_dim=time_embed_dim,
                    num_res_blocks=num_res_blocks,
                    use_attention=use_attn,
                    dropout=dropout,
                )
            )
            ch_in = ch_out
            # Pop remaining skip channels for this level (ResBlock outputs)
            for _ in range(num_res_blocks):
                channels.pop()
        
        # --- Final output ---
        self.final_norm = nn.GroupNorm(min(32, ch_in), ch_in)
        self.final_act = nn.SiLU()
        self.final_conv = nn.Conv2d(ch_in, image_channels, kernel_size=3, padding=1)
        
        # Initialize final conv to zero
        nn.init.zeros_(self.final_conv.weight)
        nn.init.zeros_(self.final_conv.bias)
    
    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
        class_label: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (B, image_channels, H, W) noisy input image
            t: (B,) integer timesteps
            class_label: (B,) optional integer class labels
        Returns:
            (B, image_channels, H, W) predicted noise (same shape as input)
        """
        # Compute time embedding
        time_emb = self.time_embedding(t)              # (B, time_embed_dim)
        
        # Add class embedding if provided
        if self.class_embedding is not None and class_label is not None:
            class_emb = self.class_embedding(class_label)  # (B, time_embed_dim)
            time_emb = time_emb + class_emb                # (B, time_embed_dim)
        
        # Initial convolution
        x = self.initial_conv(x)                       # (B, base_channels, H, W)
        
        # Encoder: collect all skip connections
        all_skips = []
        for down_block in self.down_blocks:
            x, skips = down_block(x, time_emb)
            all_skips.extend(skips)
        
        # Middle
        x = self.middle_res1(x, time_emb)              # (B, C, H', W')
        x = self.middle_attn(x)                        # (B, C, H', W')
        x = self.middle_res2(x, time_emb)              # (B, C, H', W')
        
        # Decoder: consume skip connections in reverse
        for up_block in self.up_blocks:
            x = up_block(x, all_skips, time_emb)
        
        # Final output
        x = self.final_act(self.final_norm(x))         # (B, base_channels, H, W)
        x = self.final_conv(x)                         # (B, image_channels, H, W)
        
        return x


# --- Test the complete UNet ---
torch.manual_seed(42)
model = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    attention_resolutions=(7,),
    dropout=0.0,
    num_classes=None,
)

# Test forward pass
x_test = torch.randn(2, 1, 28, 28)   # (2, 1, 28, 28) MNIST-sized input
t_test = torch.randint(0, 1000, (2,)) # (2,) random timesteps

with torch.no_grad():
    output = model(x_test, t_test)     # (2, 1, 28, 28)

print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {output.shape}")
print(f"Input == Output shape: {x_test.shape == output.shape}")

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size:           {total_params * 4 / 1024 / 1024:.1f} MB (float32)")

## 4.9 -- Parameter Count Analysis

Understanding where parameters live in the U-Net helps with architecture design. In diffusion U-Nets, the majority of parameters are concentrated in the deeper (higher-channel) layers. Attention blocks add significant parameters at the resolutions where they are placed.

### Breakdown by Module

In [ ]:
def count_params(module: nn.Module) -> int:
    """Count total parameters in a module."""
    return sum(p.numel() for p in module.parameters())


def param_breakdown(model: UNet) -> None:
    """Print a detailed parameter breakdown of the UNet by component."""
    sections = {
        "Time embedding": model.time_embedding,
        "Initial conv": model.initial_conv,
    }
    
    # Class embedding (if present)
    if model.class_embedding is not None:
        sections["Class embedding"] = model.class_embedding
    
    # Down blocks
    for i, block in enumerate(model.down_blocks):
        sections[f"DownBlock {i+1}"] = block
    
    # Middle
    middle_params = (
        count_params(model.middle_res1) +
        count_params(model.middle_attn) +
        count_params(model.middle_res2)
    )
    sections["Middle (res+attn+res)"] = None  # Handle separately
    
    # Up blocks
    for i, block in enumerate(model.up_blocks):
        sections[f"UpBlock {i+1}"] = block
    
    # Final
    final_params = (
        count_params(model.final_norm) +
        count_params(model.final_conv)
    )
    sections["Final (norm+conv)"] = None  # Handle separately
    
    total = count_params(model)
    
    print(f"{'Component':<25} {'Parameters':>12} {'% of Total':>10}")
    print("-" * 50)
    
    for name, module in sections.items():
        if module is not None:
            n = count_params(module)
        elif "Middle" in name:
            n = middle_params
        elif "Final" in name:
            n = final_params
        else:
            continue
        pct = 100.0 * n / total
        print(f"{name:<25} {n:>12,} {pct:>9.1f}%")
    
    print("-" * 50)
    print(f"{'TOTAL':<25} {total:>12,} {'100.0%':>10}")
    
    # Summary by stage
    encoder_params = sum(count_params(b) for b in model.down_blocks) + count_params(model.initial_conv)
    decoder_params = sum(count_params(b) for b in model.up_blocks) + final_params
    
    print(f"\n{'Stage Summary':}")
    print(f"  Encoder (init + down):  {encoder_params:>10,}  ({100*encoder_params/total:.1f}%)")
    print(f"  Middle:                 {middle_params:>10,}  ({100*middle_params/total:.1f}%)")
    print(f"  Decoder (up + final):   {decoder_params:>10,}  ({100*decoder_params/total:.1f}%)")
    print(f"  Time/class embedding:   {count_params(model.time_embedding):>10,}  "
          f"({100*count_params(model.time_embedding)/total:.1f}%)")


param_breakdown(model)

# Test with class-conditional model
print("\n" + "=" * 50)
print("Class-conditional UNet (10 MNIST classes):\n")

model_cond = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    attention_resolutions=(7,),
    num_classes=10,
)

# Verify class conditioning works
x_cond = torch.randn(2, 1, 28, 28)    # (2, 1, 28, 28)
t_cond = torch.randint(0, 1000, (2,))  # (2,)
labels = torch.tensor([3, 7])          # (2,) class labels

with torch.no_grad():
    out_cond = model_cond(x_cond, t_cond, class_label=labels)  # (2, 1, 28, 28)

print(f"Class-conditional output shape: {out_cond.shape}")
param_breakdown(model_cond)

### Exercise 4.9: Design a UNet Under a Parameter Budget

Design a UNet with **fewer than 1 million parameters** that still works for MNIST (28x28, 1 channel). You can adjust `base_channels`, `channel_mults`, `num_res_blocks`, and `attention_resolutions`. Verify it produces the correct output shape.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

# Strategy: reduce base_channels to 32, use fewer mults, 1 ResBlock per level, no attention
model_small = UNet(
    image_channels=1,
    base_channels=32,
    channel_mults=(1, 2, 2),
    num_res_blocks=1,
    attention_resolutions=(),  # No attention to save parameters
    dropout=0.0,
    num_classes=None,
)

x_small = torch.randn(2, 1, 28, 28)       # (2, 1, 28, 28)
t_small = torch.randint(0, 1000, (2,))     # (2,)

with torch.no_grad():
    out_small = model_small(x_small, t_small)  # (2, 1, 28, 28)

total_small = sum(p.numel() for p in model_small.parameters())
print(f"Small UNet output: {out_small.shape}")
print(f"Parameter count:   {total_small:,}")
print(f"Under 1M budget:   {total_small < 1_000_000}")
print(f"Model size:        {total_small * 4 / 1024 / 1024:.2f} MB (float32)")
print()
param_breakdown(model_small)

## Capstone: End-to-End UNet Verification

Final verification: run a complete forward pass through the full UNet, printing shapes at every stage. This confirms the entire pipeline works correctly on MNIST-sized inputs and that skip connections are properly matched between encoder and decoder.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this : Verbose forward pass showing shapes at every stage

def verbose_forward(model: UNet, x: torch.Tensor, t: torch.Tensor,
                    class_label: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Run the UNet forward pass with detailed shape logging."""
    print("=" * 60)
    print("VERBOSE FORWARD PASS")
    print("=" * 60)
    
    print(f"\nInput:       {x.shape}")
    print(f"Timesteps:   {t.shape} -> values: {t.tolist()}")
    
    # Time embedding
    time_emb = model.time_embedding(t)                    # (B, time_embed_dim)
    print(f"\nTime embedding: {time_emb.shape}")
    
    if model.class_embedding is not None and class_label is not None:
        class_emb = model.class_embedding(class_label)    # (B, time_embed_dim)
        time_emb = time_emb + class_emb
        print(f"Class embedding added: {class_emb.shape}")
    
    # Initial conv
    x = model.initial_conv(x)                             # (B, base_channels, H, W)
    print(f"\nAfter initial conv: {x.shape}")
    
    # Encoder
    print(f"\n--- ENCODER ---")
    all_skips = []
    for i, down_block in enumerate(model.down_blocks):
        x, skips = down_block(x, time_emb)
        all_skips.extend(skips)
        print(f"DownBlock {i+1}: output {x.shape} | "
              f"skips: {[s.shape for s in skips]}")
    
    print(f"\nTotal skip connections stored: {len(all_skips)}")
    
    # Middle
    print(f"\n--- MIDDLE ---")
    x = model.middle_res1(x, time_emb)
    print(f"After middle ResBlock 1: {x.shape}")
    x = model.middle_attn(x)
    print(f"After middle Attention:  {x.shape}")
    x = model.middle_res2(x, time_emb)
    print(f"After middle ResBlock 2: {x.shape}")
    
    # Decoder
    print(f"\n--- DECODER ---")
    for i, up_block in enumerate(model.up_blocks):
        skips_before = len(all_skips)
        x = up_block(x, all_skips, time_emb)
        skips_consumed = skips_before - len(all_skips)
        print(f"UpBlock {i+1}: output {x.shape} | "
              f"consumed {skips_consumed} skips | "
              f"{len(all_skips)} remaining")
    
    # Final
    print(f"\n--- OUTPUT ---")
    x = model.final_act(model.final_norm(x))
    print(f"After final norm+act: {x.shape}")
    x = model.final_conv(x)
    print(f"After final conv:     {x.shape}")
    
    assert len(all_skips) == 0, f"Unconsumed skips remaining: {len(all_skips)}"
    print(f"\nAll skip connections consumed: True")
    print("=" * 60)
    
    return x


# Run verbose forward pass on the full model
torch.manual_seed(42)
model_capstone = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    attention_resolutions=(7,),
    dropout=0.0,
    num_classes=10,
)

x_cap = torch.randn(2, 1, 28, 28)        # (2, 1, 28, 28) MNIST batch
t_cap = torch.tensor([50, 900])           # (2,) one early, one late timestep
labels_cap = torch.tensor([3, 7])         # (2,) class labels

model_capstone.eval()
with torch.no_grad():
    output_cap = verbose_forward(model_capstone, x_cap, t_cap, class_label=labels_cap)

print(f"\nFinal verification:")
print(f"  Input shape:  (2, 1, 28, 28)")
print(f"  Output shape: {output_cap.shape}")
print(f"  Shapes match: {output_cap.shape == (2, 1, 28, 28)}")
print(f"  Total params: {sum(p.numel() for p in model_capstone.parameters()):,}")

## Export: Save UNet to `utils/unet.py`

The cell below writes all U-Net components to a self-contained Python file that later modules can import directly.

In [ ]:
unet_code = '''\
"""U-Net architecture for diffusion models.

Self-contained module with all building blocks:
- SinusoidalTimestepEmbedding
- ResBlock (with time conditioning)
- AttentionBlock (spatial self-attention)
- DownBlock / UpBlock
- UNet (complete model)

Exported from module_04_unet.ipynb.
"""

import math
from typing import Optional, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class SinusoidalTimestepEmbedding(nn.Module):
    """Maps integer timesteps to sinusoidal positional embeddings.

    Args:
        embedding_dim: dimension of the output embedding vector
    """

    def __init__(self, embedding_dim: int):
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """
        Args:
            timesteps: (B,) integer tensor of timesteps
        Returns:
            embeddings: (B, embedding_dim) float tensor
        """
        half_dim = self.embedding_dim // 2
        frequencies = torch.exp(
            -math.log(10000.0) * torch.arange(half_dim, device=timesteps.device) / half_dim
        )  # (half_dim,)
        angles = timesteps[:, None].float() * frequencies[None, :]  # (B, half_dim)
        embeddings = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B, embedding_dim)
        return embeddings


class ResBlock(nn.Module):
    """Residual block with time conditioning and dropout.

    Structure: GroupNorm -> SiLU -> Conv -> +time -> GroupNorm -> SiLU -> Dropout -> Conv + residual

    Args:
        in_channels: input feature channels
        out_channels: output feature channels
        time_embed_dim: dimension of the time embedding vector
        dropout: dropout rate applied before the second convolution
    """

    def __init__(self, in_channels: int, out_channels: int, time_embed_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups=min(32, in_channels), num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=min(32, out_channels), num_channels=out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_embed_dim, out_channels),
        )

        self.residual_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

        nn.init.zeros_(self.conv2.weight)
        nn.init.zeros_(self.conv2.bias)

    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, in_channels, H, W)
            time_emb: (B, time_embed_dim)
        Returns:
            (B, out_channels, H, W)
        """
        residual = self.residual_proj(x)        # (B, out_channels, H, W)
        x = self.act(self.norm1(x))             # (B, in_channels, H, W)
        x = self.conv1(x)                       # (B, out_channels, H, W)
        t = self.time_proj(time_emb)            # (B, out_channels)
        x = x + t[:, :, None, None]             # (B, out_channels, H, W)
        x = self.act(self.norm2(x))             # (B, out_channels, H, W)
        x = self.dropout(x)                     # (B, out_channels, H, W)
        x = self.conv2(x)                       # (B, out_channels, H, W)
        return x + residual                     # (B, out_channels, H, W)


class AttentionBlock(nn.Module):
    """Spatial self-attention block with residual connection.

    Args:
        channels: number of input (and output) channels
        num_heads: number of attention heads
    """

    def __init__(self, channels: int, num_heads: int = 1):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        assert channels % num_heads == 0
        self.head_dim = channels // num_heads

        self.norm = nn.GroupNorm(num_groups=min(8, channels), num_channels=channels)
        self.to_qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)

        nn.init.zeros_(self.proj_out.weight)
        nn.init.zeros_(self.proj_out.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W) with self-attention applied + residual
        """
        B, C, H, W = x.shape
        residual = x                                                      # (B, C, H, W)
        x = self.norm(x)                                                  # (B, C, H, W)
        qkv = self.to_qkv(x)                                              # (B, 3C, H, W)
        qkv = qkv.reshape(B, 3, self.num_heads, self.head_dim, H * W)    # (B, 3, heads, hd, n)
        qkv = qkv.permute(1, 0, 2, 4, 3)                                 # (3, B, heads, n, hd)
        q, k, v = qkv[0], qkv[1], qkv[2]                                 # each: (B, heads, n, hd)
        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale              # (B, heads, n, n)
        attn = F.softmax(attn, dim=-1)                                    # (B, heads, n, n)
        out = torch.matmul(attn, v)                                       # (B, heads, n, hd)
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)                # (B, C, H, W)
        out = self.proj_out(out)                                          # (B, C, H, W)
        return out + residual                                             # (B, C, H, W)


class DownBlock(nn.Module):
    """Encoder block: N ResBlocks + optional attention + downsample.

    Args:
        in_channels: input channels
        out_channels: output channels
        time_embed_dim: time embedding dimension
        num_res_blocks: number of ResBlocks
        use_attention: whether to apply attention after each ResBlock
        dropout: dropout rate
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        time_embed_dim: int,
        num_res_blocks: int = 2,
        use_attention: bool = False,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.res_blocks = nn.ModuleList()
        self.attn_blocks = nn.ModuleList()
        for i in range(num_res_blocks):
            ch_in = in_channels if i == 0 else out_channels
            self.res_blocks.append(ResBlock(ch_in, out_channels, time_embed_dim, dropout))
            self.attn_blocks.append(
                AttentionBlock(out_channels) if use_attention else nn.Identity()
            )
        self.downsample = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1)

    def forward(
        self, x: torch.Tensor, time_emb: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        skips = []
        for res_block, attn_block in zip(self.res_blocks, self.attn_blocks):
            x = res_block(x, time_emb)
            x = attn_block(x)
            skips.append(x)
        x = self.downsample(x)
        skips.append(x)
        return x, skips


class UpBlock(nn.Module):
    """Decoder block: upsample + N ResBlocks with skip concat + optional attention.

    Args:
        in_channels: channels from deeper level
        out_channels: desired output channels
        skip_channels: channels from encoder skip
        time_embed_dim: time embedding dimension
        num_res_blocks: number of ResBlocks
        use_attention: whether to apply attention
        dropout: dropout rate
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        skip_channels: int,
        time_embed_dim: int,
        num_res_blocks: int = 2,
        use_attention: bool = False,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        )
        self.res_blocks = nn.ModuleList()
        self.attn_blocks = nn.ModuleList()
        for i in range(num_res_blocks):
            ch_in = (out_channels + skip_channels) if i == 0 else out_channels
            self.res_blocks.append(ResBlock(ch_in, out_channels, time_embed_dim, dropout))
            self.attn_blocks.append(
                AttentionBlock(out_channels) if use_attention else nn.Identity()
            )

    def forward(
        self, x: torch.Tensor, skips: List[torch.Tensor], time_emb: torch.Tensor
    ) -> torch.Tensor:
        x = self.upsample(x)
        for i, (res_block, attn_block) in enumerate(zip(self.res_blocks, self.attn_blocks)):
            skip = skips.pop()
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
            if i == 0:
                x = torch.cat([x, skip], dim=1)
            else:
                x = x + skip
            x = res_block(x, time_emb)
            x = attn_block(x)
        return x


class UNet(nn.Module):
    """Complete U-Net for diffusion models with time and optional class conditioning.

    Args:
        image_channels: input/output image channels (1 for grayscale, 3 for RGB)
        base_channels: base channel count
        channel_mults: per-level channel multipliers
        num_res_blocks: ResBlocks per encoder/decoder level
        attention_resolutions: spatial resolutions where attention is applied
        dropout: dropout rate
        num_classes: enables class-conditional generation if set
    """

    def __init__(
        self,
        image_channels: int = 1,
        base_channels: int = 64,
        channel_mults: Tuple[int, ...] = (1, 2, 4),
        num_res_blocks: int = 2,
        attention_resolutions: Tuple[int, ...] = (7,),
        dropout: float = 0.0,
        num_classes: Optional[int] = None,
    ):
        super().__init__()
        self.image_channels = image_channels
        self.num_classes = num_classes
        time_embed_dim = base_channels * 4

        # Time embedding
        self.time_embedding = nn.Sequential(
            SinusoidalTimestepEmbedding(base_channels),
            nn.Linear(base_channels, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim),
        )

        # Optional class embedding
        if num_classes is not None:
            self.class_embedding = nn.Embedding(num_classes + 1, time_embed_dim)
        else:
            self.class_embedding = None

        # Initial conv
        self.initial_conv = nn.Conv2d(image_channels, base_channels, kernel_size=3, padding=1)

        # Encoder
        self.down_blocks = nn.ModuleList()
        channels = [base_channels]
        ch_in = base_channels
        current_res = 28

        for level, mult in enumerate(channel_mults):
            ch_out = base_channels * mult
            use_attn = (current_res // 2) in attention_resolutions
            self.down_blocks.append(
                DownBlock(ch_in, ch_out, time_embed_dim, num_res_blocks, use_attn, dropout)
            )
            ch_in = ch_out
            current_res = current_res // 2
            channels.extend([ch_out] * num_res_blocks + [ch_out])

        # Middle
        self.middle_res1 = ResBlock(ch_in, ch_in, time_embed_dim, dropout)
        self.middle_attn = AttentionBlock(ch_in)
        self.middle_res2 = ResBlock(ch_in, ch_in, time_embed_dim, dropout)

        # Decoder
        self.up_blocks = nn.ModuleList()
        for level in reversed(range(len(channel_mults))):
            mult = channel_mults[level]
            ch_out = base_channels * mult
            skip_ch = channels.pop()
            decoder_res = 28 // (2 ** level)
            use_attn = decoder_res in attention_resolutions
            self.up_blocks.append(
                UpBlock(ch_in, ch_out, skip_ch, time_embed_dim, num_res_blocks, use_attn, dropout)
            )
            ch_in = ch_out
            for _ in range(num_res_blocks):
                channels.pop()

        # Final output
        self.final_norm = nn.GroupNorm(min(32, ch_in), ch_in)
        self.final_act = nn.SiLU()
        self.final_conv = nn.Conv2d(ch_in, image_channels, kernel_size=3, padding=1)
        nn.init.zeros_(self.final_conv.weight)
        nn.init.zeros_(self.final_conv.bias)

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
        class_label: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (B, image_channels, H, W) noisy input
            t: (B,) integer timesteps
            class_label: (B,) optional integer class labels
        Returns:
            (B, image_channels, H, W) predicted noise
        """
        time_emb = self.time_embedding(t)
        if self.class_embedding is not None and class_label is not None:
            time_emb = time_emb + self.class_embedding(class_label)

        x = self.initial_conv(x)

        all_skips = []
        for down_block in self.down_blocks:
            x, skips = down_block(x, time_emb)
            all_skips.extend(skips)

        x = self.middle_res1(x, time_emb)
        x = self.middle_attn(x)
        x = self.middle_res2(x, time_emb)

        for up_block in self.up_blocks:
            x = up_block(x, all_skips, time_emb)

        x = self.final_act(self.final_norm(x))
        x = self.final_conv(x)
        return x
'''

# Write to file
import os
export_path = os.path.join('utils', 'unet.py')
export_path = os.path.join('utils', 'unet.py')

with open(export_path, "w") as f:
    f.write(unet_code)

print(f"UNet exported to: {export_path}")

# Verify import works
import importlib.util
spec = importlib.util.spec_from_file_location("unet", export_path)
unet_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(unet_module)

# Test the exported module
test_model = unet_module.UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4))
test_x = torch.randn(1, 1, 28, 28)
test_t = torch.tensor([100])
with torch.no_grad():
    test_out = test_model(test_x, test_t)
print(f"Exported UNet test: {test_x.shape} -> {test_out.shape}")
print(f"Import successful: True")